In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub
import pandas as pd
import os
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

d_path  = os.path.join(path, 'Q1_data.csv')
df= pd.read_csv(d_path)
df.head()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
df.shape

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# target distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'],inplace=True)
df.columns

In [ ]:
df.head()

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
df.Delivery_Time.isnull().sum()

In [ ]:
df['Delivery_Time'].dropna(inplace=True)

In [ ]:
df.Delivery_Time.isna().sum()

In [ ]:
df.Delivery_Time.unique()

In [ ]:
df.dropna(subset=['Delivery_Time'],inplace=True)

In [ ]:
df.Delivery_Time.unique()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['Weather'],inplace=True)
df.Weather.isnull().sum()

In [ ]:
#Since the data set is small so we cant drop the missing values, so we will impute thim
#df['col'].fillna(df['col'].mean())

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
# Encode features and target using OneHotEncoder
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here:
# Standardize features using StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")# we dont scale the target
print(numerical_cols)
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# Task 5: Write your code here:

In [ ]:
# Task 1: Write your code here:
# split features from targets
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
random_forest = RandomForestRegressor(n_estimators=200)

In [ ]:
df.Delivery_Time.isnull().sum()

In [ ]:
# Task 2,3,4,5: Write your code here:

#Since we're regression problem we use kfold
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
mae_error = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):#it's return the start index for all the fold train and index for the test folds
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  random_forest.fit(X_train, y_train)
  # Validate
  y_pred = random_forest.predict(X_test)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)
  print(f"MAE: {mae}")

  # Store results
  mae_error.append(mae)

In [ ]:
df.columns

In [ ]:
# Task 2: Write your code here:
random_forest_importance = list(zip(X.columns, random_forest.feature_importances_))
sorted_random_forest_importance = sorted(random_forest_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_random_forest_importance)

# Plot feature importances
plt.figure(figsize=(20, 20))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('random_forest Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
# Attack distribution
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Deleviry time predicted')
plt.xlabel('Deleviry time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Attack distribution
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30,edgecolor='black', color='orange')
plt.title('Deleviry time Actual')
plt.xlabel('Actual Deleviry time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: